Author: Krish

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

Note: Assumes data has been restricted only to datasets that included the 'Termination Reason' column 

In [2]:
data_path = "/Users/viviadams/Downloads/CAR_Includes_Termination"

### User input ends

### Reading all filenames in the data folder

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [4]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
   # print(i, f.stem, df.shape) - removed printing file name

### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [5]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [6]:
# Checking the datatype of all columns
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [7]:
# Creating a new column 'hour' as it will be useful to visualize peak calling hours
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

## Data Cleaning & Analysis 

In [9]:
# Finding Contact Session IDs for calls that enter LegalMenu1 
legal_menu_calls = df_main.loc[df_main['Activity Name'] == 'LegalMenu1', 'Contact Session ID']

# Filtering DF to only include those Contact Session IDs 
legal_menu_calls = df_main.loc[df_main['Contact Session ID'].isin(legal_menu_calls), :]

In [10]:
# Adding Duration Column 
first_last_times = legal_menu_calls.groupby("Contact Session ID")["Activity Start Timestamp"].agg(["first", "last"])
first_last_times["Duration"] = first_last_times["last"] - first_last_times["first"]

legal_menu_calls = legal_menu_calls.merge(first_last_times["Duration"], on="Contact Session ID", how="left")


# Utilizing Meera's data cleaning 

menu_selection = legal_menu_calls.copy() 

#Give each session ID an ID number in order of appearance
menu_selection["Call ID"] = menu_selection["Contact Session ID"].map(
    {id_: i+1 for i, id_ in enumerate(menu_selection["Contact Session ID"].unique())}
)

#Calculate Time Difference between rows 
menu_selection["Time Difference"] = (
    menu_selection.groupby("Contact Session ID")["Activity Start Timestamp"]
    .diff()
    .dt.total_seconds()  
)

menu_selection = menu_selection[
    [
        'Call ID',
        'Contact Session ID',
        'EP Name',
        'Flow Name',
        'Activity Name',
        'Queue Name',
        'Agent Name', 
        'Termination Reason',
        'Activity Start Timestamp',
        'Time Difference', # seconds
        'Duration'
    ]
]

menu_selection['Time Difference'] = menu_selection['Time Difference'].fillna(0)

### Function For Identifying Positive Outcomes

In [60]:
def successful_call(df, id_col = 'Call ID', term_col='Termination Reason', term_val1='Customer Left',
               term_val2 = 'Agent Left', agent_col='Agent Name'):
    
    # Checks if row has 'Agent Left' as termination reason, or 'Customer Left as termination reason and a non-NaN agent name 
    mask = (
        (df[term_col].isin([term_val1]) &  ~df[agent_col].isna()) | df[term_col].isin([term_val2])
    )
    
    # Filtering dataframe for rows where conditions are met
        # Storing 'Call IDs' as an array 
    qualifying_calls = df.loc[mask, id_col].unique()

    # Using array to filter dataframe for those 
    valid_calls = df.loc[df[id_col].isin(qualifying_calls), id_col].dropna().unique()

    return valid_calls

### ADAPT Menu Outcomes

In [61]:
# Filtering dataframe to only include Contact Session IDs of calls that have 'ADAPTQueue' as an activity name 
adapt_calls = menu_selection.loc[menu_selection['Activity Name'] == 'ADAPTQueue', 'Contact Session ID'] 

adapt_queue = menu_selection.loc[menu_selection['Contact Session ID'].isin(adapt_calls), :]

# Finding total calls in ADAPT Queue 
print('Total Number of Calls in ADAPT Queue:', adapt_queue['Call ID'].nunique())

# Finding number of positive outcomes for ADAPT menu using successful_call function  
adapt_success = successful_call(adapt_queue)

print('Number of Positive Outcomes:', len(adapt_success)) 


Total Number of Calls in ADAPT Queue: 68
Number of Positive Outcomes: 60


In [13]:
# Filtering ADAPT dataframe to exclude Call IDs of calls with positive outcomes 
adapt_unsuccessful = adapt_queue.loc[~adapt_queue['Call ID'].isin(adapt_success)]

# Dropping the final row for calls if the Termination Reason is NaN 
last_row = adapt_unsuccessful.groupby('Call ID').cumcount(ascending=False) == 0
no_term_reason = adapt_unsuccessful['Termination Reason'].isna()
adapt_unsuccessful = adapt_unsuccessful[~(last_row & no_term_reason)]

# Grouping last Termination Reason by Call ID
adapt_unsuccessful_calls = pd.DataFrame(adapt_unsuccessful.groupby('Call ID')['Termination Reason'].last())

# Checking if Termination Reason is in list of negative outcomes 
adapt_negative = adapt_unsuccessful_calls[adapt_unsuccessful_calls['Termination Reason'].isin(['Customer Left', 'Queue Timeout', 'AGENT_UNAVAILABLE',
                                                                                   'AGENT_BUSY’', 'NO_ANSWER_FROM_AGENT', 'MEDIA_MANAGER_INTERNAL_ERROR', 'CHANNEL_FAILURE'])]

# Printing number of negative and unexplained outcomes
print('Number of Negative Outcomes:', len(adapt_negative)) 
print('Unexplained Outcomes:', len(adapt_unsuccessful_calls) - len(adapt_negative))

Number of Negative Outcomes: 8
Unexplained Outcomes: 0


In [14]:
# Identifying types of Termination Reasons in the negative outcomes 
adapt_unsuccessful_calls['Termination Reason'].value_counts()

Termination Reason
Customer Left    8
Name: count, dtype: int64

In [58]:
# Finding proportion of positive & negative outcomes for ADAPT menu 
print('Proportion of Positive ADAPT Call Outcomes:', round(len(adapt_success)/adapt_queue['Call ID'].nunique(), 2))
print('Proportion of Negative ADAPT Call Outcomes:', round(len(adapt_negative)/adapt_queue['Call ID'].nunique(), 2) )

Proportion of Positive ADAPT Call Outcomes: 0.88
Proportion of Negative ADAPT Call Outcomes: 0.12


### Consumer Menu Outcomes

In [16]:
# Filtering dataframe to only include Contact Session IDs of calls that have 'ConsumerQueue' as an activity name 
consumer_calls = menu_selection.loc[menu_selection['Activity Name'] == 'ConsumerQueue', 'Contact Session ID'] 

consumer_queue = menu_selection.loc[menu_selection['Contact Session ID'].isin(consumer_calls), :]

# Finding total calls in Consumer Queue 
print('Total Number of Calls in Consumer Queue:', consumer_queue['Call ID'].nunique())

# Finding number of positive outcomes for Consumer menu using successful_call function  
consumer_success = successful_call(consumer_queue)

print('Number of Positive Outcomes:', len(consumer_success)) 

Total Number of Calls in Consumer Queue: 701
Number of Positive Outcomes: 617


In [17]:
# Filtering Consumer dataframe to exclude Call IDs of calls with positive outcomes 
unsuccessful_consumer = consumer_queue.loc[~consumer_queue['Call ID'].isin(consumer_success)]

# Dropping the final row for calls if the Termination Reason is NaN 
last_row = unsuccessful_consumer.groupby('Call ID').cumcount(ascending=False) == 0
no_term_reason = unsuccessful_consumer['Termination Reason'].isna()
unsuccessful_consumer = unsuccessful_consumer[~(last_row & no_term_reason)]

# Grouping last Termination Reason by Call ID
consumer_unsuccessful_calls = pd.DataFrame(unsuccessful_consumer.groupby('Call ID')['Termination Reason'].last())

# Checking if Termination Reason is in list of negative outcomes 
consumer_negative = consumer_unsuccessful_calls[consumer_unsuccessful_calls['Termination Reason'].isin(['Customer Left', 'Queue Timeout', 'AGENT_UNAVAILABLE',
                                                                                   'AGENT_BUSY’', 'NO_ANSWER_FROM_AGENT', 'MEDIA_MANAGER_INTERNAL_ERROR', 'CHANNEL_FAILURE'])]

# Printing number of negative and unexplained outcomes
print('Number of Negative Outcomes:', len(consumer_negative)) 
print('Unexplained Outcomes:', len(consumer_unsuccessful_calls) - len(consumer_negative))

Number of Negative Outcomes: 84
Unexplained Outcomes: 0


In [18]:
# Identifying types of Termination Reasons in the negative outcomes 
consumer_negative['Termination Reason'].value_counts()

Termination Reason
Customer Left                   78
MEDIA_MANAGER_INTERNAL_ERROR     6
Name: count, dtype: int64

In [59]:
# Finding proportion of positive & negative outcomes for Consumer menu 
print('Proportion of Positive Outcomes:', round(len(consumer_success) / consumer_queue['Call ID'].nunique(), 2))
print('Proportion of Negative Outcomes:', round(len(consumer_negative) / consumer_queue['Call ID'].nunique(), 2)
)

Proportion of Positive Outcomes: 0.88
Proportion of Negative Outcomes: 0.12


### Criminal Menu Outcomes 

In [20]:
# Filtering dataframe to only include Contact Session IDs of calls that have 'CriminalRecordsVoicemailTransfer' as an activity name 
criminal_calls = menu_selection.loc[menu_selection['Activity Name'] == 'CriminalRecordsVoicemailTransfer', 'Contact Session ID'] 

criminal_queue = menu_selection.loc[menu_selection['Contact Session ID'].isin(criminal_calls), :]

# Finding total calls in Criminal Records Menu 
print('Total Number of Calls in Criminal Records Menu:', criminal_queue['Call ID'].nunique())

# Finding number of positive outcomes for Criminal Records Voicemail Transfer 
crim_success = criminal_queue[criminal_queue['Termination Reason'].isin(['Agent Left'])]

print('Number of Positive Outcomes:', crim_success['Call ID'].nunique()) 

Total Number of Calls in Criminal Menu: 709
Number of Positive Outcomes: 602


In [65]:
# Filtering Criminal Records dataframe to exclude Call IDs of calls with positive outcomes 
unsuccessful_criminal = criminal_queue.loc[~criminal_queue['Call ID'].isin(crim_success['Call ID'])]

# Dropping the final row for calls if the Termination Reason is NaN 
last_row = unsuccessful_criminal.groupby('Call ID').cumcount(ascending=False) == 0
no_term_reason = unsuccessful_criminal['Termination Reason'].isna()
unsuccessful_criminal = unsuccessful_criminal[~(last_row & no_term_reason)]

# Grouping last Termination Reason by Call ID
crim_neutral_calls = pd.DataFrame(unsuccessful_criminal.groupby('Call ID')['Termination Reason'].last())


# Checking if Termination Reason is in list of negative outcomes 
crim_neutral = crim_neutral_calls[crim_neutral_calls['Termination Reason'].isin(['Customer Left'])]

# Printing number of neutral and unexplained outcomes
print('Number of Neutral Outcomes:', len(crim_neutral))
print('Number of Unexplained Outcomes:', len(crim_neutral_calls) - len(crim_neutral))

Number of Neutral Outcomes: 107
Number of Unexplained Outcomes: 0


In [22]:
# Finding proportion of positive & neutral outcomes for ADAPT menu 
print('Proportion of Positive Outcomes:', round(len(crim_success) / criminal_queue['Call ID'].nunique(), 2))
print('Proportion of Neutral Outcomes:', round(len(crim_neutral) / criminal_queue['Call ID'].nunique(), 2))

Proportion of Positive Outcomes: 0.85
Proportion of Neutral Outcomes: 0.15


 ### Unexplained Legal Menu 2 Calls

In [56]:
# Filtering original dataframe (calls through Legal Menu 1) to focus on Legal Menu 2 
lm2_calls = menu_selection.loc[menu_selection['Activity Name'] == 'LegalMenu2', 'Call ID']

# Removing each of the above sub-menu's calls from the datafra,e 
pass_through = menu_selection.loc[menu_selection['Call ID'].isin(lm2_calls)]

not_covered1 = pass_through.loc[~pass_through['Call ID'].isin(criminal_queue['Call ID'])]

not_covered2 = not_covered1.loc[~not_covered1['Call ID'].isin(adapt_queue['Call ID'])]

not_covered3 = not_covered2.loc[~not_covered2['Call ID'].isin(consumer_queue['Call ID'])]

# Outcomes that are not covered by this analysis -> calls that do not enter a queue within LegalMenu2
not_covered3['Call ID'].nunique()

30587

### Front Desk Transfer Outcomes

In [25]:
# Filtering dataframe to only include Contact Session IDs of calls that have 'FrontDeskTransfer' as an activity name 
front_desk_calls = menu_selection.loc[menu_selection['Activity Name'] == 'FrontDeskTransfer', 'Contact Session ID'] 

front_desk_queue = menu_selection.loc[menu_selection['Contact Session ID'].isin(front_desk_calls), :]

# Finding total calls in Front Desk Transfer
print('Total Number of Calls for Front Desk Transfer:', front_desk_queue['Call ID'].nunique())

# Finding number of positive outcomes for Front Desk Transfer using successful_call function  
front_desk_success = successful_call(front_desk_queue)

print('Number of Positive Outcomes:', len(front_desk_success)) 

Total Number of Calls for Front Desk Transfer: 795
Number of Positive Outcomes: 794


In [26]:
# Filtering Front Desk Transfer dataframe to exclude Call IDs of calls with positive outcomes 
unsuccessful_fd = front_desk_queue.loc[~front_desk_queue['Call ID'].isin(front_desk_success)]

# Dropping the final row for calls if the Termination Reason is NaN 
last_row = unsuccessful_fd.groupby('Call ID').cumcount(ascending=False) == 0
no_term_reason = unsuccessful_fd['Termination Reason'].isna()
unsuccessful_fd = unsuccessful_fd[~(last_row & no_term_reason)]

# Grouping last Termination Reason by Call ID
unsuccessful_fd_calls = pd.DataFrame(unsuccessful_fd.groupby('Call ID')['Termination Reason'].last())

# Checking if Termination Reason is in list of negative outcomes 
negative_fd_calls = unsuccessful_fd_calls[unsuccessful_fd_calls['Termination Reason'].isin(['Customer Left', 'Queue Timeout', 'AGENT_UNAVAILABLE','AGENT_BUSY’', 
                                                                                            'NO_ANSWER_FROM_AGENT', 'MEDIA_MANAGER_INTERNAL_ERROR', 'CHANNEL_FAILURE'])]

# Printing number of negative and unexplained outcomes
print('Number of Negative Outcomes:', len(negative_fd_calls)) 
print('Number of Unexplained Outcomes:', len(unsuccessful_fd_calls) - len(negative_fd_calls))

Number of Negative Outcomes: 1
Number of Unexplained Outcomes: 0


In [66]:
# Identifying types of Termination Reasons in the negative outcomes 
negative_fd_calls['Termination Reason'].value_counts()

Termination Reason
Customer Left    1
Name: count, dtype: int64

In [27]:
# Finding proportion of positive & negative outcomes for ADAPT menu 
print('Proportion of Positive Outcomes:', round(len(front_desk_success) / front_desk_queue['Call ID'].nunique(), 3))
print('Proportion of Negative Outcomes:', round(len(negative_fd_calls) / front_desk_queue['Call ID'].nunique(), 3))

Proportion of Positive Outcomes: 0.999
Proportion of Negative Outcomes: 0.001


### Outcomes for Overall Legal Menu 

In [82]:
# Finding Total Calls that Pass through Legal Menu 1 
print('Total Calls through Legal Menu 1:', menu_selection['Call ID'].nunique())

# Filtering dataframe to only include calls through Legal Menu 2 
legal_menu2_calls = df_main.loc[df_main['Activity Name'] == 'LegalMenu2', 'Contact Session ID']
legal_menu2_calls = df_main.loc[df_main['Contact Session ID'].isin(legal_menu2_calls), :]
print('Total Calls through Legal Menu 2:',legal_menu2_calls['Contact Session ID'].nunique())

# Finding the calls that reach their terminal node in Legal Menu 2 
terminal_node_calls = consumer_queue['Call ID'].nunique() +  adapt_queue['Call ID'].nunique() + criminal_queue['Call ID'].nunique()
print('Calls with Terminal Nodes in Legal Menu 2:', terminal_node_calls)
print('Roughly', round(terminal_node_calls / legal_menu2_calls['Contact Session ID'].nunique(), 3), 'of calls through Legal Menu 2 have their terminal node within the menu')


Total Calls through Legal Menu 1: 33222
Total Calls through Legal Menu 2: 32065
Calls with Terminal Nodes in Legal Menu 2: 1478
Roughly 0.046 of calls through Legal Menu 2 have their terminal node within the menu


In [81]:
# Finding overall number & proportion of positive outcomes for Legal Menu (1 & 2)
positive_outcomes_overall = len(adapt_success) + len(consumer_success) + len(crim_success) + len(front_desk_success)

print('Overall Legal Menu Calls with a Positive Outcome:', positive_outcomes_overall) 
print('Proportion of Overall Calls:', round(positive_outcomes_overall /  menu_selection['Call ID'].nunique(), 4))

Overall Legal Menu Calls with a Positive Outcome: 2073
Proportion of Overall Calls: 0.0624


In [80]:
# Finding overall number & proportion of negative outcomes for Legal Menu (1 & 2)
negative_outcomes_overall = len(adapt_negative) + len(consumer_negative) + len(negative_fd_calls)

print('Overall Legal Menu Calls with a Negative Outcome:', negative_outcomes_overall) 
print('Proportion of Overall Calls:', round(negative_outcomes_overall /  menu_selection['Call ID'].nunique(), 4))

Overall Legal Menu Calls with a Negative Outcome: 93
Proportion of Overall Calls: 0.0028


### Call Outcomes for Legal Menu 2

In [30]:
# Finding overall number of positive outcomes for Legal Menu 2
positive_outcomes2 = len(adapt_success) + len(consumer_success) + len(crim_success) 

print('Legal Menu 2 Calls with a Positive Outcome:',  positive_outcomes2)

# Proportion including calls that terminate in another menu 
print('Proportion of Overall Legal Menu 2 Calls:', round(positive_outcomes2/  legal_menu2_calls['Contact Session ID'].nunique(), 3))

# Proportion of calls that terminate in Legal Menu 2 
print('Proportion of Calls Terminating in Legal Menu 2:',round(positive_outcomes2/ terminal_node_calls, 3) )


Legal Menu 2 Calls with a Positive Outcome: 1279
Proportion of Overall Legal Menu 2 Calls: 0.04
Proportion of Calls Terminating in Legal Menu 2: 0.563


In [31]:
# Finding overall number of negative outcomes for Legal Menu 2
negative_outcomes2 = len(adapt_negative) + len(consumer_negative) 

print('Legal Menu 2 Calls with a Negative Outcome:',  negative_outcomes2)

# Proportion including calls that terminate in another menu 
print('Proportion of Overall Legal Menu 2 Calls:', round(negative_outcomes2/  legal_menu2_calls['Contact Session ID'].nunique(), 3))

# Proportion of calls that terminate in Legal Menu 2 
print('Proportion of Calls Terminating in Legal Menu 2:',round(negative_outcomes2/ terminal_node_calls, 3) )


Legal Menu 2 Calls with a Negative Outcome: 92
Proportion of Overall Legal Menu 2 Calls: 0.003
Proportion of Calls Terminating in Legal Menu 2: 0.04


In [68]:
# Neutral Outcomes for Legal Menu 2 

print('Legal Menu 2 Calls with a Negative Outcome:',  len(crim_neutral))

# Proportion including calls that terminate in another menu 
print('Proportion of Overall Legal Menu 2 Calls:', round(len(crim_neutral)/  legal_menu2_calls['Contact Session ID'].nunique(), 3))

# Proportion of calls that terminate in Legal Menu 2 
print('Proportion of Calls Terminating in Legal Menu 2:',round(len(crim_neutral)/ terminal_node_calls, 3) )

Legal Menu 2 Calls with a Negative Outcome: 107
Proportion of Overall Legal Menu 2 Calls: 0.003
Proportion of Calls Terminating in Legal Menu 2: 0.072


### Outcomes Accounted for by this Analysis

In [77]:
assigned_outcomes = (terminal_node_calls + front_desk_queue['Call ID'].nunique()) / menu_selection['Call ID'].nunique()

print('Approximately',  round(assigned_outcomes*100, 2), '% of calls through were assigned an outcome through this analysis, but this includes all calls that terminate with Legal Menu 1 or 2')

Approximately 6.84 % of calls through were assigned an outcome through this analysis, but this includes all calls that terminate with Legal Menu 1 or 2
